In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!pip install rdkit==2024.9.5
!pip install torch_geometric==2.5.3

In [ ]:
import os
import sys
import torch
from torch.utils.data import DataLoader
doc_name = "/content/drive/MyDrive/HeckLit-Code-Colab"
sys.path.append(doc_name)
from utils.rxn import *
from utils.molecule import *
from utils.dataset_analysis import *
from models.DeepLearnModel import *
import time
from tqdm import tqdm
import datetime
import warnings
warnings.filterwarnings("ignore")

In [ ]:
# 1. import data
data = pd.read_excel("%s/data/Heck/Heck_fp.xlsx" % doc_name)
random_state = 3
data = data.sample(random_state=random_state, frac=1).reset_index(drop=True)

# 2. build dataset & dataloader
rxn_dataset = list()
rxn_list = df_to_rxn_list(data)
len_drfp = 0
yield_dict = list()

for batch in tqdm(range(data.shape[0])):
    rxn = rxn_list[batch]

    # features
    drfp = torch.tensor(read_drfp(data.loc[batch]["drfp"]), dtype=torch.float32)
    len_drfp = drfp.shape[0]
    # label
    y = rxn.rxn_yield / 100

    rxn_dataset.append([drfp, y])
    yield_dict.append(y * 100)

# yield shot split
split_num = 100
yield_dict = shot_classifier(yield_dict, split_num, upper=150, lower=50)

# split of train & test set
ratio = 0.8
batch_size = 1500
batch = len(rxn_dataset)
train_set = rxn_dataset[0: int(ratio * batch)]
test_set = rxn_dataset[int(ratio * batch) + 1:]

# data_loader
train_loader = DataLoader(train_set, batch_size=batch_size, shuffle=True)
test_loader = DataLoader(test_set, batch_size=batch_size, shuffle=True)

train_R2 = list()
train_RMSE = list()
train_MAE = list()
test_R2 = list()
test_RMSE = list()
test_MAE = list()
pred = list()
true = list()

# 3. training of the model
# params
t = 2000
lr = 1e-4

# model
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
fds = None
model = ANN(input_size=len_drfp, FDS=fds).to(device)
opti = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=1e-5)
criterion = nn.MSELoss()

# Training
# best performance
best = [0, 0, 0, 0, 0, 0, [], []]  # train_R2, train_RMSE, train_MAE, test_R2, test_RMSE, test_MAE, test_predict, test_true

for epoch in tqdm(range(t)):
    # Training
    global_loss = torch.tensor([0.])

    for data in train_loader:
        x = data[:-1][0].to(device)
        y = torch.unsqueeze(data[-1], dim=1).to(device)
        loss = criterion(model.forward(x, y, epoch, device).float(), y.float())
        opti.zero_grad()
        loss.backward()
        opti.step()
        global_loss += loss.item()

    # record of loss during training
    # performance in train set
    with torch.no_grad():
        pred = list()
        true = list()
        for data in train_loader:
            x = data[:-1][0].to(device)
            tr = torch.unsqueeze(data[-1], dim=1).to(device)
            pr = list(model.forward(x, tr, epoch, device).cpu().detach().numpy())
            pred += pr
            true += list(tr.cpu().detach().numpy())
        train_R2.append(R2(np.array(pred), np.array(true)))
        train_RMSE.append(RMSE(np.array(pred), np.array(true)))
        train_MAE.append(MAE(np.array(pred), np.array(true)))

    # performance in test set
    with torch.no_grad():
        pred = list()
        true = list()
        for data in test_loader:
            x = data[:-1][0].to(device)
            tr = torch.unsqueeze(data[-1], dim=1).to(device)
            pr = list(model.forward(x, tr, epoch, device).cpu().detach().numpy())
            pred += pr
            true += list(tr.cpu().detach().numpy())
        test_R2.append(R2(np.array(pred), np.array(true)))
        test_RMSE.append(RMSE(np.array(pred), np.array(true)))
        test_MAE.append(MAE(np.array(pred), np.array(true)))

        if epoch == 0 or test_R2[-1] >= best[3]:
            best = [train_R2[-1], train_RMSE[-1], train_MAE[-1], test_R2[-1], test_RMSE[-1], test_MAE[-1], pred, true]

# 4.Evaluation
# Performance in differrent shot
fewshot_pr = list()
fewshot_tr = list()
medshot_pr = list()
medshot_tr = list()
manyshot_pr = list()
manyshot_tr = list()

tr = best[-1]
pr = best[-2]
for i in range(len(tr)):
    for key in yield_dict.keys():
        if key <= tr[i] * 100 < key + 100 / split_num:
            cls = yield_dict[key][1]

            if cls == "Few-shot":
                fewshot_pr.append(pr[i])
                fewshot_tr.append(tr[i])
            if cls == "Medium-shot":
                medshot_pr.append(pr[i])
                medshot_tr.append(tr[i])
            if cls == "Many-shot":
                manyshot_pr.append(pr[i])
                manyshot_tr.append(tr[i])

RMSE_fewshot = RMSE(np.array(fewshot_pr), np.array(fewshot_tr))
MAE_fewshot = MAE(np.array(fewshot_tr), np.array(fewshot_pr))
RMSE_medshot = RMSE(np.array(medshot_pr), np.array(medshot_tr))
MAE_medshot = MAE(np.array(medshot_tr), np.array(medshot_pr))
RMSE_manyshot = RMSE(np.array(manyshot_pr), np.array(manyshot_tr))
MAE_manyshot = MAE(np.array(manyshot_tr), np.array(manyshot_pr))

# Performance in train set
print("R2 of train set is:%.3f+-%f\tbest:%f\n" % (
np.array(train_R2[-10:]).mean(), np.array(train_R2[-10:]).std(), best[0]))
print("RMSE of train set is: %.3f+-%f\tbest:%f\n" % (
np.array(train_RMSE[-10:]).mean(), np.array(train_RMSE[-10:]).std(), best[1]))

# Performance in test set
print("R2 of test set is:%.3f+-%.3f\tbest:%f\n" % (
np.array(test_R2[-10:]).mean(), np.array(test_R2[-10:]).std(), best[3]))
print("RMSE of test set is: %.3f+-%f\tbest:%f\n" % (
np.array(test_RMSE[-10:]).mean(), np.array(test_RMSE[-10:]).std(), best[4]))


# 5.Figure
import matplotlib.pyplot as plt
import seaborn as sns

shot_color = {
    "Few-shot":"#F47F72",
    "Medium-shot":"#8DD2C5",
    "Many-shot":"#7FB2D5"
}

fig = plt.figure(dpi=500, figsize=(16, 6))
# Yield Distribution
plt.subplot(1, 2, 1)
type = [[], [], []]
for key in yield_dict.keys():
    color = shot_color[yield_dict[key][1]]
    b = plt.bar(x=int(key), height=yield_dict[key][0], width=100/split_num, color=color)

    if yield_dict[key][1] == "Few-shot":
        type[0] = b
    if yield_dict[key][1] == "Medium-shot":
        type[1] = b
    if yield_dict[key][1] == "Many-shot":
        type[2] = b

plt.title("Yield Distribution", fontsize=18)
plt.xlabel("Reaction Number", fontsize=14)
plt.ylabel("Yield(%)", fontsize=14)
plt.xticks([0, 50, 100])
plt.legend(type, ["Few-shot", "Medium-shot", "Many-shot"], loc="upper left", prop={'size': 13})

# Error Distrubution
type = [[], [], []]
plt.subplot(1, 2, 2)
# Few-shot
for i in range(len(fewshot_tr)):
    rmse = RMSE(fewshot_pr[i], fewshot_tr[i])
    few = plt.bar(x=int(fewshot_tr[i]*100), height=rmse, width=100/split_num, color=shot_color["Few-shot"])
    type[0] = few
# Medium-shot
for i in range(len(medshot_tr)):
    rmse = RMSE(medshot_pr[i], medshot_tr[i])
    med = plt.bar(x=int(medshot_tr[i]*100), height=rmse, width=100/split_num, color=shot_color["Medium-shot"])
    type[1] = med
# Many-shot
for i in range(len(manyshot_tr)):
    rmse = RMSE(manyshot_pr[i], manyshot_tr[i])
    many = plt.bar(x=int(manyshot_tr[i]*100), height=rmse, width=100/split_num, color=shot_color["Many-shot"])
    type[2] = many

plt.legend(type, ["Few-shot", "Medium-shot", "Many-shot"], loc="upper left", prop={'size': 13})
plt.xlabel("Observed Yield(%)", fontsize=14)
plt.ylabel("RMSE Value", fontsize=14)
plt.xticks([0, 50, 100])
plt.title("Error Distrubution", fontsize=18)

plt.tight_layout()
plt.savefig("%s/figures/ShotErrorPlot.png" % doc_name)
plt.show()